# Structured Output from LLMs

## Introduction

LLMs can be instructed to produce **structured output** that maps to a specific schema. This is useful for:
- Extracting structured data from unstructured text
- Building reliable downstream applications that parse LLM output
- Data validation and type safety

[LlamaIndex](https://www.llamaindex.ai) provides built-in support for [structured outputs](https://developers.llamaindex.ai/python/framework/module_guides/querying/structured_outputs/) using Pydantic models.

## Packages

We use `llama_index` with the Ollama integration for local LLM inference.

In [ ]:
import os
from pydantic import BaseModel, Field
from typing import List

from llama_index.llms.ollama import Ollama
from llama_index.core.llms import ChatMessage, MessageRole


## Defining the Schema

We define the output structure using [Pydantic](https://docs.pydantic.dev/) models. The LLM will be constrained to produce output that conforms to this schema.

In [ ]:
class Work(BaseModel):
    """A creative work (movie, book, etc.)"""
    title: str
    year: int

class Character(BaseModel):
    """A fictional character with their attributes"""
    name: str
    age: int
    appears_in: List[Work] = Field(..., description="A list of works the character appears in")
    fact: List[str] = Field(..., description="A list of facts about the character")
    powers: List[str] = Field(..., description="A list of powers the character has")


The `as_structured_llm()` method wraps the LLM to produce output conforming to our Pydantic model. The response object's `.raw` attribute contains the parsed model instance.

In [ ]:
from ai_dojo.lm import get_ollama_url

# Create base LLM
llm = Ollama(
    model="olmo-3:7b",
    request_timeout=240.0,
    base_url=get_ollama_url(),
    json_mode=True,  # Enable JSON mode for structured output
)

# Wrap as structured LLM that outputs Character objects
structured_llm = llm.as_structured_llm(Character)


In [ ]:
response = structured_llm.complete("Tell me about Gandalf")

# The response.raw is already a Character Pydantic object
character: Character = response.raw

In [ ]:
from rich import print as rprint


In [ ]:
rprint(response)

In [ ]:
rprint(character)

### Using the Chat Interface

You can also use the chat interface with structured output:

In [ ]:
messages = [
    ChatMessage(
        role=MessageRole.SYSTEM,
        content="You are a helpful assistant that provides information about fictional characters.",
    ),
    ChatMessage(
        role=MessageRole.USER,
        content="Tell me about Gandalf from Lord of the Rings",
    ),
]

response = structured_llm.chat(messages)
character: Character = response.raw


In [ ]:
rprint(character)

## Extracting Structured Data from Text

A key application of structured output is **information extraction**: parsing unstructured text into structured records. Here we extract event details from announcement texts.

In [ ]:
from datetime import datetime

class Event(BaseModel):
    """An event with structured details"""
    title: str
    date: datetime
    location: str

event_texts = [
    "Join us for the Annual Tech Meetup on March 15th, 2025 at the Downtown Convention Center.",
    "The Python Workshop will be held on 2025-04-22 in Room 101, Science Building.",
    "Don't miss the AI Conference happening December 3, 2025 at Grand Hotel Ballroom.",
]

extractor = llm.as_structured_llm(Event)

events = []
for text in event_texts:
    response = extractor.complete(f"Extract event details from this text:\n{text}")
    events.append(response.raw)

In [ ]:
for event in events:
    rprint(event)

<div style="border:2px solid gray; padding:10px; width: 95%;">
<b>🫳 Hands-on: Extract recipes from text</b>

Design a Pydantic model to represent a recipe (consider: name, ingredients, amounts...) and extract structured data from the texts below.

Hint: You can use **nested models** to represent hierarchical data.
</div>

In [ ]:
recipe_texts = [
    "Classic Pancakes: Mix 2 cups flour, 1 tablespoon sugar, 2 eggs, and 1.5 cups milk.",
    "Simple Vinaigrette: Whisk together 3 tbsp olive oil, 1 tbsp vinegar, 1 tsp mustard, salt to taste.",
    "Hot Chocolate: Combine 2 cups milk, 3 tablespoons cocoa powder, and 2 tablespoons sugar.",
]

In [ ]:
# Define your Pydantic model(s) here


In [ ]:
# Extract recipes and display results


---
_This notebook is licensed under a [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)](https://creativecommons.org/licenses/by-nc-sa/4.0/). Copyright © 2024-2026 [Christian Staudt](https://clstaudt.me)_